# 🧠 Contrastive Multi-Modal Learning for Early Alzheimer's Risk Detection
### GPU-Accelerated | RTX 4060 Optimized | ADNI Dataset

**Architecture:**
- 3D CNN Encoder (MRI) + MLP Encoder (Tabular) → NT-Xent Contrastive Fusion
- XGBoost + Shallow Neural Network ensemble
- SHAP + Grad-CAM explainability
- Mixed-precision (FP16) for RTX 4060 speed

**File Layout Expected:**
```
C:\Users\Rathish K\Documents\ML\v1\
  Dataset\
    ADNIMERGE_09Mar2026.csv
    DXSUM_09Mar2026.csv
    FAQ_09Mar2026.csv
    MEDHIST_09Mar2026.csv
    MMSE_09Mar2026.csv
    NPI_09Mar2026.csv
    VITALS_09Mar2026.csv
    MRI_metadata_with_VISCODE.csv
    ADNI_T1_Baseline_MRI\   ← NIfTI .nii/.nii.gz files
```

## CELL 1 — Install Dependencies (Run Once)

In [1]:
# Run this cell ONCE. Comment it out after first successful run.
import subprocess, sys

pkgs = [
    # Core deep learning — PyTorch with CUDA 11.8 (works on RTX 4060)
    'torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118',
    # Neuroimaging
    'nibabel',
    'nilearn',
    'SimpleITK',
    # ML
    'scikit-learn',
    'xgboost',
    'imbalanced-learn',
    # Explainability
    'shap',
    'grad-cam',          # pytorch-grad-cam
    # Utils
    'pandas numpy scipy matplotlib seaborn tqdm openpyxl',
    'antspyx',           # optional: best N4 bias correction; skip if install fails
]

for pkg in pkgs:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkg.split(),
        capture_output=True
    )
    status = '✅' if result.returncode == 0 else '⚠️ (check manually)'
    print(f'{status}  {pkg.split()[0]}')

print('\n✅ Installation complete.')

✅  torch
✅  nibabel
✅  nilearn
✅  SimpleITK
✅  scikit-learn
✅  xgboost
✅  imbalanced-learn
✅  shap
✅  grad-cam
✅  pandas
✅  antspyx

✅ Installation complete.


## CELL 2 — GPU Setup & Imports

In [2]:
import os, gc, json, warnings, random, time, pickle
from pathlib import Path
from collections import Counter
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import ndimage

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

import nibabel as nib
import SimpleITK as sitk

# ── PyTorch (PRIMARY framework — better GPU support than TF on Windows RTX)
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast   # mixed precision FP16

import sklearn
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)
from sklearn.impute import KNNImputer
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import shap
from tqdm import tqdm

# ─────────────────────────────────────────────────────────────────────────────
#  GPU CONFIGURATION  (RTX 4060 — Zephyrus Q14 2024)
# ─────────────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'🚀 GPU detected  : {gpu_name}')
    print(f'   VRAM          : {gpu_mem:.1f} GB')
    print(f'   CUDA version  : {torch.version.cuda}')
    # Enable TF32 for faster matmul on Ampere (RTX 40-series)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True    # auto-tune convolutions
else:
    print('⚠️  No GPU found — running on CPU (will be slow)')

print(f'\n🔧 PyTorch version : {torch.__version__}')
print(f'🔧 Device          : {DEVICE}')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

🚀 GPU detected  : NVIDIA GeForce RTX 4060 Laptop GPU
   VRAM          : 8.6 GB
   CUDA version  : 12.1

🔧 PyTorch version : 2.5.1+cu121
🔧 Device          : cuda


## CELL 3 — Project Configuration & Paths

In [26]:
# ─────────────────────────────────────────────────────────────────────────────
#  ⚙️  CONFIGURE THESE PATHS TO YOUR LOCAL MACHINE
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR    = Path(r'C:\Users\Rathish K\Documents\ML\v1')
DATA_DIR    = Path(r'C:\Users\Rathish K\Documents\ML\Dataset')
MRI_DIR     = DATA_DIR / 'ADNI_T1_Baseline_MRI'    # folder with .nii/.nii.gz files
OUTPUT_DIR  = BASE_DIR / 'outputs'
MODEL_DIR   = BASE_DIR / 'models'
PLOT_DIR    = BASE_DIR / 'plots'
CACHE_DIR   = BASE_DIR / 'cache'

for d in [OUTPUT_DIR, MODEL_DIR, PLOT_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ─────────────────────────────────────────────────────────────────────────────
#  Hyperparameters  (tuned for RTX 4060 8GB VRAM)
# ─────────────────────────────────────────────────────────────────────────────
# MRI: downsample to 64x64x64 — fast enough on RTX 4060, keeps spatial info
MRI_TARGET_SHAPE   = (64, 64, 64)
BATCH_SIZE         = 4          # safe for 8GB VRAM with 3D-CNN
EPOCHS             = 60
LR                 = 1e-4
PROJECTION_DIM     = 128
N_CLASSES          = 4
CLASS_NAMES        = ['CN', 'EMCI', 'LMCI', 'AD']
CONTRASTIVE_TEMP   = 0.07
PATIENCE           = 12        # early stopping
USE_AMP            = True      # mixed precision — speeds up RTX 4060 by ~2x
NUM_WORKERS        = 0         # DataLoader workers (Windows: keep ≤ 2)

# CSV filenames
CSV_FILES = {
    'merge'  : r'C:\\Users\\Rathish K\Documents\\ML\Dataset\ADNIMERGE_09Mar2026.csv',
    'dx'     : r'C:\\Users\\Rathish K\Documents\\ML\Dataset\DXSUM_09Mar2026.csv',
    'faq'    : r'C:\\Users\\Rathish K\Documents\\ML\Dataset\\FAQ_09Mar2026.csv',
    'medhist': r'C:\\Users\\Rathish K\Documents\\ML\Dataset\\MEDHIST_09Mar2026.csv',
    'mmse'   : r'C:\\Users\\Rathish K\Documents\\ML\Dataset\\MMSE_09Mar2026.csv',
    'npi'    : r'C:\\Users\\Rathish K\Documents\\ML\Dataset\\NPI_09Mar2026.csv',
    'vitals' : r'C:\\Users\\Rathish K\Documents\\ML\Dataset\\VITALS_09Mar2026.csv',
    'mri_meta': r'C:\\Users\\Rathish K\Documents\\ML\Dataset\\MRI_metadata_with_VISCODE.csv',
}

CACHE_PATH = CACHE_DIR / 'mri_volumes_64x64x64.npz'

print('✅ Configuration complete.')
print(f'   Base   : {BASE_DIR}')
print(f'   MRI    : {MRI_DIR}')
print(f'   Cache  : {CACHE_PATH}')
print(f'   Device : {DEVICE} | AMP: {USE_AMP}')

✅ Configuration complete.
   Base   : C:\Users\Rathish K\Documents\ML\v1
   MRI    : C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI
   Cache  : C:\Users\Rathish K\Documents\ML\v1\cache\mri_volumes_64x64x64.npz
   Device : cuda | AMP: True


## CELL 4 — Load & Merge All Tabular CSV Data

In [4]:
def load_csv(name, filename, data_dir):
    path = data_dir / filename
    if not path.exists():
        print(f'  ⚠️  {filename} not found. Skipping.')
        return None
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip().upper() for c in df.columns]
    print(f'  ✅ {name:10s} → {df.shape[0]:6,} rows × {df.shape[1]} cols')
    return df

print('📂 Loading CSV datasets...')
raw = {k: load_csv(k, v, DATA_DIR) for k, v in CSV_FILES.items()}
print(f'\n✅ Loaded {sum(v is not None for v in raw.values())} / {len(CSV_FILES)} datasets')

📂 Loading CSV datasets...
  ✅ merge      → 16,421 rows × 116 cols
  ✅ dx         → 16,092 rows × 41 cols
  ✅ faq        → 13,398 rows × 27 cols
  ✅ medhist    →  3,083 rows × 39 cols
  ✅ mmse       → 14,719 rows × 58 cols
  ✅ npi        →  8,357 rows × 168 cols
  ✅ vitals     → 17,368 rows × 26 cols
  ✅ mri_meta   →  1,071 rows × 4 cols

✅ Loaded 8 / 8 datasets


In [5]:
print(type(raw))
print(raw.keys())
print(raw['merge'])

<class 'dict'>
dict_keys(['merge', 'dx', 'faq', 'medhist', 'mmse', 'npi', 'vitals', 'mri_meta'])
        RID COLPROT ORIGPROT        PTID  SITE VISCODE    EXAMDATE DX_BL  \
0         2   ADNI1    ADNI1  011_S_0002    11      bl  2005-09-08    CN   
1         3   ADNI1    ADNI1  011_S_0003    11      bl  2005-09-12    AD   
2         3   ADNI1    ADNI1  011_S_0003    11     m06  2006-03-13    AD   
3         3   ADNI1    ADNI1  011_S_0003    11     m12  2006-09-12    AD   
4         3   ADNI1    ADNI1  011_S_0003    11     m24  2007-09-12    AD   
...     ...     ...      ...         ...   ...     ...         ...   ...   
16416  4349   ADNI3    ADNI2  018_S_4349    18    m138  2023-03-30    CN   
16417  6801   ADNI3    ADNI3  041_S_6801    41     m42  2023-06-30   SMC   
16418  5097   ADNI3    ADNI2  041_S_5097    41    m126  2023-08-16   SMC   
16419  6515   ADNI3    ADNI3  007_S_6515     7     m60  2023-08-24   SMC   
16420  5214   ADNI3    ADNI2  126_S_5214   126    m120  2023-09-06 

## CELL 5 — Tabular Feature Engineering & Label Creation

In [61]:
print('🔧 Building master feature table...')

# ── 5A: ADNIMERGE baseline ────────────────────────────────────────────────
merge = raw['merge'].copy()
if 'VISCODE' in merge.columns:
    merge = merge[merge['VISCODE'] == 'bl'].copy()

KEEP = [
    'RID', 'PTID', 'AGE', 'PTGENDER', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT',
    'PTMARRY', 'APOE4',
    'FDG', 'AV45', 'ABETA', 'TAU', 'PTAU',
    'CDRSB', 'ADAS11', 'ADAS13', 'ADASQ4',
    'MMSE', 'MOCA', 'FAQ',
    'RAVLT_IMMEDIATE', 'RAVLT_LEARNING', 'RAVLT_FORGETTING', 'RAVLT_PERC_FORGETTING',
    'LDELTOTAL', 'DIGITSCOR', 'TRABSCOR',
    'ECOGPTMEM', 'ECOGPTLANG', 'ECOGPTVISSPAT', 'ECOGPTPLAN',
    'ECOGSPMEM', 'ECOGSPPLAN', 'ECOGSPTOTAL',
    'VENTRICLES', 'HIPPOCAMPUS', 'WHOLEBRAIN', 'ENTORHINAL', 'FUSIFORM', 'MIDTEMP', 'ICV',
    'DX_BL', 'DX',
]
KEEP = [c for c in KEEP if c in merge.columns]
merge = merge[KEEP].drop_duplicates(subset=['RID']).reset_index(drop=True)
print(f'  ADNIMERGE baseline: {merge.shape}')

# ── 5B: Label mapping ─────────────────────────────────────────────────────
DX_MAP = {
    'CN':'CN', 'EMCI':'EMCI', 'LMCI':'LMCI', 'AD':'AD',
    'MCI':'LMCI', 'SMC':'CN', 'DEMENTIA':'AD',
    'NL':'CN', 'MCI TO DEMENTIA':'AD', 'NL TO MCI':'EMCI',
}
dx_col = 'DX' if 'DX' in merge.columns else 'DX_BL'
merge['LABEL'] = merge[dx_col].apply(
    lambda v: DX_MAP.get(str(v).strip().upper(), np.nan) if pd.notna(v) else np.nan
)
merge = merge.dropna(subset=['LABEL']).copy()
print(f'  Full label distribution:\n{merge["LABEL"].value_counts().to_string()}')

# ── KEY FIX: Detect ONLY the classes actually present in the data ──────────
# ADNI T1-baseline MRI only matched CN/EMCI/AD subjects (no LMCI in 163)
# We build CLASS_NAMES from ONLY present labels, in canonical order.
# This gives contiguous 0-based integers that XGBoost, SNN, Fusion all agree on.
_CANONICAL = ['CN', 'EMCI', 'LMCI', 'AD']
_present   = set(merge['LABEL'].unique())
CLASS_NAMES = [c for c in _CANONICAL if c in _present]
N_CLASSES   = len(CLASS_NAMES)

le = LabelEncoder()
le.fit(CLASS_NAMES)  # fits only on present classes → 0-based contiguous
merge['LABEL_INT'] = le.transform(merge['LABEL'])

print(f'\n  ✅ Active classes ({N_CLASSES}): {CLASS_NAMES}')
print(f'  Label mapping : {dict(zip(CLASS_NAMES, range(N_CLASSES)))}')

# ── 5C: Merge VITALS ──────────────────────────────────────────────────────
if raw['vitals'] is not None:
    v = raw['vitals'].copy()
    vital_num = [c for c in v.columns if v[c].dtype in [np.float64, np.int64] and c != 'RID']
    vagg = v.groupby('RID')[vital_num].mean().reset_index()
    merge = merge.merge(vagg, on='RID', how='left', suffixes=('', '_V'))
    print(f'  After VITALS merge: {merge.shape}')

# ── 5D: Merge MMSE ────────────────────────────────────────────────────────
if raw['mmse'] is not None:
    m = raw['mmse'].copy()
    mmse_num = [c for c in m.columns if m[c].dtype in [np.float64, np.int64] and c not in ['RID','PHASE']]
    magg = m.groupby('RID')[mmse_num].mean(numeric_only=True).reset_index()
    merge = merge.merge(magg, on='RID', how='left', suffixes=('', '_MMSE'))
    print(f'  After MMSE merge  : {merge.shape}')

# ── 5E: Merge NPI ─────────────────────────────────────────────────────────
if raw['npi'] is not None:
    n = raw['npi'].copy()
    npi_num = [c for c in n.columns if n[c].dtype in [np.float64, np.int64] and c not in ['RID','PHASE']]
    nagg = n.groupby('RID')[npi_num].sum(numeric_only=True).reset_index()
    merge = merge.merge(nagg, on='RID', how='left', suffixes=('', '_NPI'))
    print(f'  After NPI merge   : {merge.shape}')

# ── 5F: Merge FAQ ─────────────────────────────────────────────────────────
if raw['faq'] is not None:
    f = raw['faq'].copy()
    faq_cols = [c for c in f.columns if c.startswith('FAQ') or c == 'RID']
    fagg = f[faq_cols].groupby('RID').first().reset_index()
    merge = merge.merge(fagg, on='RID', how='left', suffixes=('', '_FAQ'))
    print(f'  After FAQ merge   : {merge.shape}')

# ── 5G: Impute + Scale ────────────────────────────────────────────────────
DROP_META = ['PTID', 'DX_BL', 'DX', 'LABEL', 'LABEL_INT', 'RID', 'VISCODE']
DROP_META = [c for c in DROP_META if c in merge.columns]

feat_df   = merge.drop(columns=DROP_META).copy()
label_arr = merge['LABEL_INT'].values
rid_arr   = merge['RID'].values

for col in feat_df.select_dtypes(include=['object','category']).columns:
    feat_df[col] = LabelEncoder().fit_transform(feat_df[col].astype(str).fillna('missing'))

feat_df = feat_df.select_dtypes(include=[np.number])

print('  Running KNN imputation...')
imputer = KNNImputer(n_neighbors=5)
X_tab   = imputer.fit_transform(feat_df).astype(np.float32)

scaler  = StandardScaler()
X_tab   = scaler.fit_transform(X_tab).astype(np.float32)

FEATURE_NAMES = feat_df.columns.tolist()
print(f'\n✅ Final tabular feature matrix: {X_tab.shape}')
print(f'   Features: {len(FEATURE_NAMES)}')


🔧 Building master feature table...
  ADNIMERGE baseline: (2430, 44)
  Full label distribution:
LABEL
LMCI    1101
CN       895
AD       413

  ✅ Active classes (3): ['CN', 'LMCI', 'AD']
  Label mapping : {'CN': 0, 'LMCI': 1, 'AD': 2}
  After VITALS merge: (2409, 61)
  After MMSE merge  : (2409, 101)
  After NPI merge   : (2409, 258)
  After FAQ merge   : (2409, 269)
  Running KNN imputation...

✅ Final tabular feature matrix: (2409, 263)
   Features: 263


## CELL 6 — MRI File Discovery & Metadata Linking

In [7]:
print('🔍 Discovering MRI NIfTI files...')

def discover_mri_files(mri_root: Path):
    """Walk ADNI folder tree, extract PTID from path, return DataFrame."""
    records = []
    for p in sorted(mri_root.rglob('*.nii*')):
        ptid = None
        # ADNI structure: .../ADNI/<PTID>/MPR__GradWarp.../...
        for part in p.parts:
            # ADNI PTID format: XXX_S_XXXX  (e.g. 011_S_0002)
            if '_S_' in part and len(part) == 10:
                ptid = part
                break
        records.append({'PTID': ptid, 'PATH': str(p)})
    return pd.DataFrame(records)

if MRI_DIR.exists():
    mri_df = discover_mri_files(MRI_DIR)
    print(f'  Found {len(mri_df)} NIfTI files')
else:
    print(f'  ⚠️  MRI directory not found: {MRI_DIR}')
    mri_df = pd.DataFrame(columns=['PTID', 'PATH'])

# Build PTID → RID map from ADNIMERGE
adni_raw = raw['merge'].copy()
if 'PTID' in adni_raw.columns and 'RID' in adni_raw.columns:
    ptid_rid = adni_raw[['PTID','RID']].drop_duplicates().set_index('PTID')['RID'].to_dict()
    mri_df['RID'] = mri_df['PTID'].map(ptid_rid)
    mri_df = mri_df.dropna(subset=['RID'])
    mri_df['RID'] = mri_df['RID'].astype(int)
    # Keep only one scan per subject (first found)
    mri_df = mri_df.drop_duplicates(subset=['RID']).reset_index(drop=True)
    print(f'  Mapped to RID: {len(mri_df)} unique subjects')
else:
    print('  ⚠️  PTID/RID columns not found in ADNIMERGE')

print(mri_df.head(3))

🔍 Discovering MRI NIfTI files...
  Found 1071 NIfTI files
  Mapped to RID: 187 unique subjects
         PTID                                               PATH  RID
0  002_S_0413  C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T...  413
1  002_S_0559  C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T...  559
2  002_S_0729  C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T...  729


## CELL 7 — MRI Preprocessing Pipeline (GPU-Accelerated Resize)

In [12]:
import torch.nn.functional as TF
from nilearn import image as nl_image

# ─────────────────────────────────────────────────────────────────────────────
#  MNI152 Template — nilearn ships this built-in, no download needed
# ─────────────────────────────────────────────────────────────────────────────
def get_mni152_template():
    """Load the MNI152 1mm T1 template via nilearn (auto-downloaded once)."""
    from nilearn.datasets import load_mni152_template
    return load_mni152_template(resolution=2)   # 2mm = 91x109x91 voxels

MNI_TEMPLATE = get_mni152_template()
print(f'✅ MNI152 template loaded: {MNI_TEMPLATE.shape}')


def skull_strip_sitk(sitk_img: sitk.Image) -> sitk.Image:
    """
    Skull stripping via SimpleITK (no FSL/ANTs dependency needed):
      1. Otsu threshold → binary brain mask
      2. Morphological closing to fill internal holes
      3. Keep largest connected component (= brain, discard scalp patches)
      4. Apply mask to original image

    This removes skull, scalp, orbital fat, and dura mater.
    Suitable for T1-weighted ADNI baseline scans.
    """
    # Step 1: Otsu threshold
    otsu_filter = sitk.OtsuThresholdImageFilter()
    otsu_filter.SetInsideValue(0)
    otsu_filter.SetOutsideValue(1)
    brain_mask = otsu_filter.Execute(sitk_img)

    # Step 2: Morphological closing to fill holes inside brain
    closing = sitk.BinaryMorphologicalClosingImageFilter()
    closing.SetKernelRadius(3)
    closing.SetForegroundValue(1)
    brain_mask = closing.Execute(brain_mask)

    # Step 3: Keep only the largest connected component (brain)
    cc_filter = sitk.ConnectedComponentImageFilter()
    labeled   = cc_filter.Execute(brain_mask)
    relabel   = sitk.RelabelComponentImageFilter()
    relabel.SetMinimumObjectSize(10000)   # voxels; removes small noise islands
    labeled   = relabel.Execute(labeled)
    brain_mask = sitk.BinaryThreshold(labeled, 1, 1, 1, 0)  # keep label=1 only

    # Step 4: Apply mask — zero out everything outside brain
    mask_f   = sitk.Cast(brain_mask, sitk.sitkFloat32)
    stripped = sitk.Multiply(sitk_img, mask_f)
    return stripped


def register_to_mni152(nib_img: nib.Nifti1Image,
                        mni_template) -> nib.Nifti1Image:
    """
    Affine registration to MNI152 template via nilearn.
    Uses a combination of linear (affine) transformations to map
    each subject's unique brain geometry into the MNI152 common
    coordinate system (91x109x91 at 2mm resolution).

    Why this matters:
      - Aligns hippocampus, entorhinal cortex, and frontal regions
        to the same voxel positions across all subjects
      - Makes Grad-CAM heatmaps anatomically interpretable
      - Required for your base paper's MNI152 template claim
    """
    registered = nl_image.resample_to_img(
        source_img=nib_img,
        target_img=mni_template,
        interpolation='continuous',
        copy_header=False,
    )
    return registered


def load_and_preprocess_mri(path: str,
                             target_shape=MRI_TARGET_SHAPE) -> np.ndarray:
    """
    Full MRI preprocessing pipeline (updated):
      1.  Load NIfTI with nibabel
      2.  N4 bias field correction (SimpleITK) — remove MRI shading artefacts
      3.  Skull stripping (SimpleITK Otsu + morphology) — remove skull/scalp/
          orbital fat / dura mater, keeping only cerebral tissue
      4.  MNI152 affine registration (nilearn) — bring all brains into the
          same coordinate system (91x109x91 @ 2mm)
      5.  GPU resize to target_shape via trilinear interpolation
      6.  Min-max normalise to [0, 1]

    Returns: float32 numpy array of shape target_shape
    """
    # ── Step 1: Load ──────────────────────────────────────────────────────────
    nib_img  = nib.load(path)
    data     = nib_img.get_fdata(dtype=np.float32)

    # ── Step 2: N4 Bias Field Correction ──────────────────────────────────────
    sitk_img  = sitk.GetImageFromArray(data)
    sitk_img  = sitk.Cast(sitk_img, sitk.sitkFloat32)
    corrector = sitk.N4BiasFieldCorrectionImageFilter()
    corrector.SetMaximumNumberOfIterations([50, 50, 30, 20])
    sitk_img  = corrector.Execute(sitk_img)

    # ── Step 3: Skull Stripping ───────────────────────────────────────────────
    sitk_img  = skull_strip_sitk(sitk_img)

    # Rebuild nibabel image with corrected + stripped data
    stripped_data = sitk.GetArrayFromImage(sitk_img).astype(np.float32)
    # Copy original affine so registration has correct world coordinates
    nib_stripped  = nib.Nifti1Image(stripped_data, nib_img.affine)

    # ── Step 4: MNI152 Registration ───────────────────────────────────────────
    nib_mni = register_to_mni152(nib_stripped, MNI_TEMPLATE)
    data    = nib_mni.get_fdata(dtype=np.float32)

    # ── Step 5: GPU Resize to target_shape ────────────────────────────────────
    tensor = torch.from_numpy(data).unsqueeze(0).unsqueeze(0).to(DEVICE)  # (1,1,D,H,W)
    tensor = TF.interpolate(tensor,
                            size=target_shape,
                            mode='trilinear',
                            align_corners=False)
    data = tensor.squeeze().cpu().numpy()

    # ── Step 6: Min-Max Normalise ─────────────────────────────────────────────
    mn, mx = data.min(), data.max()
    if mx > mn:
        data = (data - mn) / (mx - mn)
    else:
        data = np.zeros_like(data)

    return data.astype(np.float32)

def process_all_mri(mri_df: pd.DataFrame):
    """
    Process all MRI NIfTI files listed in the requested DataFrame.
    Leverages GPU for fast resampling if DEVICE is set to cuda.
    """
    volumes = []
    rids = []
    
    for _, row in tqdm(mri_df.iterrows(), total=len(mri_df), desc='Preprocessing MRIs'):
        try:
            vol = load_and_preprocess_mri(row['PATH'])
            volumes.append(vol)
            rids.append(row['RID'])
        except Exception as e:
            print(f'⚠️ Error on {row["PATH"]}: {e}')
            
    return np.array(volumes, dtype=np.float32), np.array(rids, dtype=np.int64)

def process_all_mri(mri_df: pd.DataFrame):
    """
    Process all MRI NIfTI files listed in the requested DataFrame.
    Leverages GPU for fast resampling if DEVICE is set to cuda.
    """
    volumes = []
    rids = []
    
    # Use tqdm for a console progress bar
    for _, row in tqdm(mri_df.iterrows(), total=len(mri_df), desc='Preprocessing MRIs'):
        try:
            vol = load_and_preprocess_mri(row['PATH'])
            volumes.append(vol)
            rids.append(row['RID'])
        except Exception as e:
            print(f'⚠️ Error on {row["PATH"]}: {e}')
            
    # Return as numpy arrays compatible with saving
    return np.array(volumes, dtype=np.float32), np.array(rids, dtype=np.int64)

print('✅ MRI preprocessing pipeline ready.')
print('   Steps: N4 bias correction → Skull strip → MNI152 registration → GPU resize → Normalise')
print(f'   Target shape: {MRI_TARGET_SHAPE}')


✅ MNI152 template loaded: (99, 117, 95)
✅ MRI preprocessing pipeline ready.
   Steps: N4 bias correction → Skull strip → MNI152 registration → GPU resize → Normalise
   Target shape: (64, 64, 64)


## CELL 8 — Process or Load Cached MRI Volumes

In [13]:
if CACHE_PATH.exists():
    print(f'📦 Cache found → loading from {CACHE_PATH}')
    cache = np.load(CACHE_PATH)
    mri_volumes = cache['volumes']
    mri_rids    = cache['rids']
    print(f'  Loaded {len(mri_volumes)} volumes, shape: {mri_volumes.shape}')

elif len(mri_df) == 0:
    print('⚠️  No MRI files found. Generating SYNTHETIC volumes for pipeline testing.')
    print('   Replace with real MRI data before final evaluation.')
    N = len(merge)
    # Use simple random noise as placeholder — replace with real preprocessing
    mri_volumes = np.random.rand(N, *MRI_TARGET_SHAPE).astype(np.float32)
    mri_rids    = merge['RID'].values.astype(np.int64)

else:
    print(f'🧠 Processing {len(mri_df)} MRI scans (GPU-accelerated)...')
    print('   This will take ~15-30 min for 1000+ scans on RTX 4060.')
    mri_volumes, mri_rids = process_all_mri(mri_df)
    # Save cache for future runs
    np.savez_compressed(CACHE_PATH, volumes=mri_volumes, rids=mri_rids)
    print(f'  ✅ Saved cache → {CACHE_PATH}')

print(f'\n✅ MRI volumes shape: {mri_volumes.shape}')
print(f'   RIDs shape       : {mri_rids.shape}')

🧠 Processing 187 MRI scans (GPU-accelerated)...
   This will take ~15-30 min for 1000+ scans on RTX 4060.


Preprocessing MRIs:  13%|█▎        | 24/187 [11:54<56:57, 20.96s/it]  

⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\013_S_0996\localizer\2007-01-03_13_21_12.0\I35298\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A13BC7BB0): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 511, 511]


Preprocessing MRIs:  47%|████▋     | 87/187 [35:00<09:57,  5.97s/it]  

⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\033_S_0514\Circle_Scout\2006-06-19_14_17_24.0\I17245\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A139D7B60): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 255, 255]
⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\033_S_0724\Circle_Scout\2006-08-11_14_22_12.0\I20414\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A7E2F9480): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 255, 255]
⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI

...

⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\133_S_0629\Circle_Scout\2006-07-19_11_48_24.0\I19235\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A0C258150): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 255, 255]
⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\133_S_0638\Circle_Scout\2006-09-06_09_28_53.0\I43167\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A70269AD0): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 255, 255]
⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI

Preprocessing MRIs:  93%|█████████▎| 174/187 [1:01:57<01:36,  7.43s/it]

⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\133_S_0913\Circle_Scout\2007-01-18_09_16_02.0\I36704\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A0C242C80): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 255, 255]
⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI_T1_Baseline_MRI\ADNI\133_S_1031\Circle_Scout\2006-11-16_08_50_49.0\I29909\converted_mri.nii.gz: Exception thrown in SimpleITK N4BiasFieldCorrectionImageFilter_Execute: D:\a\SimpleITK\SimpleITK\bld\ITK-prefix\include\ITK-5.4\itkImageBase.hxx:79:
ITK ERROR: Image(0000028A7EBB2FC0): Zero-valued spacing is not supported and may result in undefined behavior.
Refusing to change spacing from [1, 1, 1] to [0, 255, 255]
⚠️ Error on C:\Users\Rathish K\Documents\ML\Dataset\ADNI

Preprocessing MRIs: 100%|██████████| 187/187 [1:06:42<00:00, 21.40s/it]


  ✅ Saved cache → C:\Users\Rathish K\Documents\ML\v1\cache\mri_volumes_64x64x64.npz

✅ MRI volumes shape: (163, 64, 64, 64)
   RIDs shape       : (163,)


## CELL 9 — Align MRI + Tabular by Subject ID (RID)

In [66]:
print('🔗 Aligning MRI volumes with tabular features on RID...')

mri_rid_idx = {rid: i for i, rid in enumerate(mri_rids)}
tab_rid_idx = {r: i for i, r in enumerate(rid_arr)}

common_rids = [r for r in rid_arr if r in mri_rid_idx]
print(f'  Subjects with both MRI + tabular data: {len(common_rids)}')

al_vols, al_tabs, al_labels = [], [], []
for rid in common_rids:
    al_vols.append(mri_volumes[mri_rid_idx[rid]])
    al_tabs.append(X_tab[tab_rid_idx[rid]])
    al_labels.append(label_arr[tab_rid_idx[rid]])

# Final arrays
X_mri = np.array(al_vols,   dtype=np.float32)   # (N, D, H, W)
X_tab_al = np.array(al_tabs, dtype=np.float32)  # (N, F)
y        = np.array(al_labels, dtype=np.int64)  # (N,)

print(f'  X_mri  : {X_mri.shape}')
print(f'  X_tab  : {X_tab_al.shape}')
print(f'  Labels : {Counter(y)}')

# Free memory — MRI volumes are large
del mri_volumes
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

🔗 Aligning MRI volumes with tabular features on RID...
  Subjects with both MRI + tabular data: 163


NameError: name 'mri_volumes' is not defined

## CELL 10 — Train / Val / Test Split + SMOTE

In [15]:
print('✂️  Splitting data (stratified, no data leakage)...')

# 15% test hold-out
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
trainval_idx, test_idx = next(sss1.split(X_mri, y))

X_mri_tv, X_mri_test   = X_mri[trainval_idx],    X_mri[test_idx]
X_tab_tv, X_tab_test   = X_tab_al[trainval_idx], X_tab_al[test_idx]
y_tv,     y_test        = y[trainval_idx],        y[test_idx]

# 15% val from remaining
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.176, random_state=SEED)
train_idx, val_idx = next(sss2.split(X_mri_tv, y_tv))

X_mri_train = X_mri_tv[train_idx];  X_mri_val = X_mri_tv[val_idx]
X_tab_train = X_tab_tv[train_idx];  X_tab_val = X_tab_tv[val_idx]
y_train     = y_tv[train_idx];      y_val     = y_tv[val_idx]

print(f'  Train : {len(y_train)} | Val: {len(y_val)} | Test: {len(y_test)}')
print(f'  Train class dist: {Counter(y_train)}')

# SMOTE on tabular training features to handle class imbalance
print('\n  Applying SMOTE...')
smote = SMOTE(random_state=SEED, k_neighbors=min(3, min(Counter(y_train).values())-1))
X_tab_train_sm, y_train_sm = smote.fit_resample(X_tab_train, y_train)
print(f'  After SMOTE: {Counter(y_train_sm)}')

print('\n✅ Split complete.')

✂️  Splitting data (stratified, no data leakage)...
  Train : 113 | Val: 25 | Test: 25
  Train class dist: Counter({np.int64(3): 48, np.int64(1): 39, np.int64(0): 26})

  Applying SMOTE...
  After SMOTE: Counter({np.int64(0): 48, np.int64(1): 48, np.int64(3): 48})

✅ Split complete.


## CELL 11 — PyTorch Dataset & DataLoader (GPU-Ready)

In [31]:
class AlzheimerDataset(Dataset):
    """Multi-modal dataset: MRI volume + tabular features + label."""

    def __init__(self, mri_arr, tab_arr, labels, augment=False):
        self.mri    = torch.from_numpy(mri_arr).unsqueeze(1)  # (N,1,D,H,W)
        self.tab    = torch.from_numpy(tab_arr)
        self.labels = torch.from_numpy(labels).long()
        self.aug    = augment

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        mri = self.mri[idx].float()
        tab = self.tab[idx].float()
        lbl = self.labels[idx]

        if self.aug:
            # Simple MRI augmentation: random flip along each axis
            for ax in [1, 2, 3]:
                if random.random() > 0.5:
                    mri = torch.flip(mri, [ax])
            # Add small Gaussian noise
            mri = mri + 0.01 * torch.randn_like(mri)
            mri = mri.clamp(0, 1)

        return mri, tab, lbl


# ── Create datasets ────────────────────────────────────────────────────────────
train_ds = AlzheimerDataset(X_mri_train, X_tab_train, y_train, augment=True)
val_ds   = AlzheimerDataset(X_mri_val,   X_tab_val,   y_val,   augment=False)
test_ds  = AlzheimerDataset(X_mri_test,  X_tab_test,  y_test,  augment=False)

# ── DataLoaders with pin_memory for faster GPU transfer ───────────────────────
PIN = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN)

print('✅ DataLoaders ready.')
print(f'  Train batches : {len(train_loader)}')
print(f'  Val batches   : {len(val_loader)}')
print(f'  Test batches  : {len(test_loader)}')

✅ DataLoaders ready.
  Train batches : 28
  Val batches   : 7
  Test batches  : 7


## CELL 12 — 3D CNN Encoder Architecture (PyTorch)

In [20]:
class ConvBlock3D(nn.Module):
    """Conv3D → BN → ReLU block."""
    def __init__(self, in_ch, out_ch, kernel=3, stride=1):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_ch, out_ch, kernel, stride=stride, padding=kernel//2, bias=False),
            nn.BatchNorm3d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)


class CNN3DEncoder(nn.Module):
    """
    Lightweight 3D-CNN optimized for RTX 4060 (8GB VRAM):
      Input: (B, 1, 64, 64, 64)
      Block1: 32 filters  → MaxPool → (B,32,32,32,32)
      Block2: 64 filters  → MaxPool → (B,64,16,16,16)
      Block3: 128 filters → MaxPool → (B,128,8,8,8)
      Block4: 256 filters → GAP     → (B,256)
      Projection head     → (B, PROJECTION_DIM) L2-normalised
    """
    def __init__(self, proj_dim=PROJECTION_DIM):
        super().__init__()
        self.block1 = nn.Sequential(
            ConvBlock3D(1, 32), ConvBlock3D(32, 32),
            nn.MaxPool3d(2), nn.Dropout3d(0.1)
        )
        self.block2 = nn.Sequential(
            ConvBlock3D(32, 64), ConvBlock3D(64, 64),
            nn.MaxPool3d(2), nn.Dropout3d(0.1)
        )
        self.block3 = nn.Sequential(
            ConvBlock3D(64, 128), ConvBlock3D(128, 128),
            nn.MaxPool3d(2)
        )
        self.block4 = nn.Sequential(
            ConvBlock3D(128, 256)
        )
        self.gap    = nn.AdaptiveAvgPool3d(1)  # → (B, 256, 1, 1, 1)
        self.proj   = nn.Sequential(
            nn.Linear(256, proj_dim * 2),
            nn.ReLU(inplace=True),
            nn.Linear(proj_dim * 2, proj_dim)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.gap(x).flatten(1)       # (B, 256)
        x = self.proj(x)                  # (B, proj_dim)
        return F.normalize(x, dim=1)      # L2 normalise

    def get_last_conv_output(self, x):
        """For Grad-CAM: returns (features, embedding)."""
        x1 = self.block1(x)
        x2 = self.block2(x1)
        x3 = self.block3(x2)
        x4 = self.block4(x3)              # last conv output
        gap = self.gap(x4).flatten(1)
        emb = F.normalize(self.proj(gap), dim=1)
        return x4, emb


cnn_enc = CNN3DEncoder().to(DEVICE)

# Quick parameter count
n_params = sum(p.numel() for p in cnn_enc.parameters() if p.requires_grad)
print(f'✅ CNN3DEncoder ready — {n_params:,} trainable parameters')

# Quick forward pass test
with torch.no_grad():
    dummy = torch.randn(2, 1, *MRI_TARGET_SHAPE).to(DEVICE)
    out   = cnn_enc(dummy)
    print(f'   Test forward pass: {dummy.shape} → {out.shape}')

del dummy; gc.collect()

✅ CNN3DEncoder ready — 1,842,784 trainable parameters
   Test forward pass: torch.Size([2, 1, 64, 64, 64]) → torch.Size([2, 128])


102

## CELL 13 — Tabular MLP Encoder with Self-Attention (PyTorch)

In [22]:
class TabularEncoder(nn.Module):
    """
    MLP + Multi-Head Self-Attention tabular encoder.
    Captures inter-feature dependencies (e.g., MMSE ↔ APOE4 interactions).
    Output: L2-normalised embedding of shape (B, proj_dim)
    """
    def __init__(self, n_features, proj_dim=PROJECTION_DIM):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(n_features, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
        )
        # Self-attention: treat the 128-d vector as 1 token of dim 128
        self.attn = nn.MultiheadAttention(embed_dim=128, num_heads=4,
                                           dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(128)
        self.proj = nn.Sequential(
            nn.Linear(128, proj_dim * 2),
            nn.ReLU(inplace=True),
            nn.Linear(proj_dim * 2, proj_dim)
        )

    def forward(self, x):
        h = self.mlp(x)                         # (B, 128)
        h_seq = h.unsqueeze(1)                   # (B, 1, 128) — single token
        h_attn, _ = self.attn(h_seq, h_seq, h_seq)
        h = self.norm(h + h_attn.squeeze(1))    # residual
        e = self.proj(h)                         # (B, proj_dim)
        return F.normalize(e, dim=1)


n_tab_feats = X_tab_train.shape[1]
tab_enc = TabularEncoder(n_tab_feats).to(DEVICE)

n_params = sum(p.numel() for p in tab_enc.parameters() if p.requires_grad)
print(f'✅ TabularEncoder ready — {n_params:,} trainable parameters')
print(f'   Input features: {n_tab_feats}')

✅ TabularEncoder ready — 233,472 trainable parameters
   Input features: 263


## CELL 14 — Multi-Modal Fusion Model + NT-Xent Contrastive Loss

In [100]:
# ── NT-Xent Contrastive Loss ───────────────────────────────────────────────────
class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross-Entropy (SimCLR style).
    Treats (z_mri_i, z_tab_i) as a positive pair; all others as negatives.
    """
    def __init__(self, temperature=CONTRASTIVE_TEMP):
        super().__init__()
        self.T = temperature

    def forward(self, z_mri, z_tab):
        B = z_mri.size(0)
        z = torch.cat([z_mri, z_tab], dim=0)          # (2B, D)
        sim = torch.mm(z, z.T) / self.T               # (2B, 2B)
        # Mask self-similarity
        mask = torch.eye(2 * B, device=z.device).bool()
        sim.masked_fill_(mask, float('-inf'))
        # Positive pairs: i ↔ i+B
        labels = torch.cat([
            torch.arange(B, 2*B, device=z.device),
            torch.arange(0, B,   device=z.device)
        ])
        return F.cross_entropy(sim, labels)


# ── Full Fusion Model ──────────────────────────────────────────────────────────
class AlzheimerFusionModel(nn.Module):
    """
    Full pipeline:
      CNN(MRI) → z_mri  ]
                          → Cross-modal Attention → Classifier → 4-class output
      MLP(Tab) → z_tab  ]

    Also exposes embeddings for contrastive training and XGBoost.
    Supports missing-modality masking.
    """
    def __init__(self, cnn_encoder, tab_encoder,
                 n_classes=N_CLASSES, proj_dim=PROJECTION_DIM):
        super().__init__()
        self.cnn_enc  = cnn_encoder
        self.tab_enc  = tab_encoder
        self.proj_dim = proj_dim

        # Cross-modal attention (MRI attends to tabular and vice-versa)
        self.cross_attn = nn.MultiheadAttention(embed_dim=proj_dim, num_heads=4,
                                                 dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(proj_dim)

        # Classifier head
        self.classifier = nn.Sequential(
            nn.Linear(proj_dim * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes)
        )

        # Learnable modality-missing replacement token
        self.mri_zero = nn.Parameter(torch.zeros(1, proj_dim))
        self.tab_zero = nn.Parameter(torch.zeros(1, proj_dim))

    def forward(self, mri=None, tab=None,
                mri_missing=False, tab_missing=False):
        B = mri.size(0) if mri is not None else tab.size(0)

        z_mri = self.cnn_enc(mri) if not mri_missing else \
                self.mri_zero.expand(B, -1)
        z_tab = self.tab_enc(tab) if not tab_missing else \
                self.tab_zero.expand(B, -1)

        # Cross-modal attention: stack as sequence of 2 tokens
        z_seq  = torch.stack([z_mri, z_tab], dim=1)   # (B, 2, proj_dim)
        z_fuse, _ = self.cross_attn(z_seq, z_seq, z_seq)
        z_fuse = self.norm(z_fuse + z_seq)             # residual
        z_flat = z_fuse.flatten(1)                     # (B, 2*proj_dim)

        return self.classifier(z_flat)

    def get_embeddings(self, mri, tab):
        return self.cnn_enc(mri), self.tab_enc(tab)


fusion_model = AlzheimerFusionModel(cnn_enc, tab_enc).to(DEVICE)
contrastive_loss_fn = NTXentLoss().to(DEVICE)

# Class weights for imbalanced loss (AD upweighted)
# Dynamic class weights — built from actual present classes
_cw_map = {'CN': 1.0, 'EMCI': 2.0, 'LMCI': 2.0, 'AD': 3.0}
class_weights = torch.tensor([_cw_map.get(c, 1.0) for c in CLASS_NAMES],
                              dtype=torch.float32).to(DEVICE)
ce_loss_fn    = nn.CrossEntropyLoss(weight=class_weights)

# Optimizer with weight decay
optimizer = torch.optim.AdamW(fusion_model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

# Mixed-precision scaler (key for RTX 4060 speed)
scaler_amp = GradScaler(enabled=USE_AMP)

total_params = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f'✅ AlzheimerFusionModel ready — {total_params:,} total trainable parameters')
print(f'   Mixed precision (FP16): {USE_AMP}')
print(f'   VRAM usage estimate   : ~{total_params * 4 / 1e9:.2f} GB (weights only)')

✅ AlzheimerFusionModel ready — 2,241,891 total trainable parameters
   Mixed precision (FP16): True
   VRAM usage estimate   : ~0.01 GB (weights only)


## CELL 15 — GPU-Accelerated Training Loop (Mixed Precision)

In [32]:
ALPHA          = 0.3    # contrastive loss weight
CHECKPOINT     = MODEL_DIR / 'fusion_best.pt'

train_loss_hist, val_loss_hist = [], []
best_val_loss  = float('inf')
patience_count = 0

print(f'🚀 Training on {DEVICE} | Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | AMP: {USE_AMP}')
print('='*65)

for epoch in range(1, EPOCHS + 1):
    # ── TRAIN ─────────────────────────────────────────────────────────────────
    fusion_model.train()
    t_losses = []

    for mri_b, tab_b, lbl_b in train_loader:
        mri_b = mri_b.to(DEVICE, non_blocking=True)
        tab_b = tab_b.to(DEVICE, non_blocking=True)
        lbl_b = lbl_b.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)  # faster than zero_grad()

        with autocast(enabled=USE_AMP):        # FP16 forward pass
            logits          = fusion_model(mri_b, tab_b)
            z_mri, z_tab    = fusion_model.get_embeddings(mri_b, tab_b)
            ce_loss         = ce_loss_fn(logits, lbl_b)
            cl_loss         = contrastive_loss_fn(z_mri, z_tab)
            total_loss      = (1 - ALPHA) * ce_loss + ALPHA * cl_loss

        scaler_amp.scale(total_loss).backward()
        scaler_amp.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), 1.0)
        scaler_amp.step(optimizer)
        scaler_amp.update()

        t_losses.append(total_loss.item())

    scheduler.step()

    # ── VALIDATE ──────────────────────────────────────────────────────────────
    fusion_model.eval()
    v_losses, v_preds, v_true = [], [], []

    with torch.no_grad(), autocast(enabled=USE_AMP):
        for mri_b, tab_b, lbl_b in val_loader:
            mri_b = mri_b.to(DEVICE, non_blocking=True)
            tab_b = tab_b.to(DEVICE, non_blocking=True)
            lbl_b = lbl_b.to(DEVICE, non_blocking=True)
            logits = fusion_model(mri_b, tab_b)
            v_losses.append(ce_loss_fn(logits, lbl_b).item())
            v_preds.extend(logits.argmax(1).cpu().numpy())
            v_true.extend(lbl_b.cpu().numpy())

    mean_t = np.mean(t_losses)
    mean_v = np.mean(v_losses)
    val_acc = accuracy_score(v_true, v_preds)
    train_loss_hist.append(mean_t)
    val_loss_hist.append(mean_v)

    if epoch % 5 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]['lr']
        print(f'  Ep {epoch:3d}/{EPOCHS} | T-Loss: {mean_t:.4f} | '
              f'V-Loss: {mean_v:.4f} | V-Acc: {val_acc:.4f} | LR: {lr_now:.2e}')

    # Early stopping
    if mean_v < best_val_loss:
        best_val_loss = mean_v
        patience_count = 0
        torch.save(fusion_model.state_dict(), CHECKPOINT)
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'\n  ⏹  Early stopping at epoch {epoch}')
            break

# Load best weights
fusion_model.load_state_dict(torch.load(CHECKPOINT, map_location=DEVICE))
print(f'\n✅ Training complete. Best val loss: {best_val_loss:.4f}')

🚀 Training on cuda | Epochs: 60 | Batch: 4 | AMP: True
  Ep   1/60 | T-Loss: 1.5427 | V-Loss: 1.1695 | V-Acc: 0.4800 | LR: 9.99e-05
  Ep   5/60 | T-Loss: 1.2959 | V-Loss: 0.9553 | V-Acc: 0.4400 | LR: 9.83e-05
  Ep  10/60 | T-Loss: 1.1768 | V-Loss: 0.8404 | V-Acc: 0.5600 | LR: 9.34e-05
  Ep  15/60 | T-Loss: 1.0234 | V-Loss: 0.7130 | V-Acc: 0.6000 | LR: 8.55e-05
  Ep  20/60 | T-Loss: 0.9812 | V-Loss: 0.6811 | V-Acc: 0.5600 | LR: 7.52e-05
  Ep  25/60 | T-Loss: 0.9916 | V-Loss: 0.5163 | V-Acc: 0.6800 | LR: 6.33e-05
  Ep  30/60 | T-Loss: 0.9401 | V-Loss: 0.5127 | V-Acc: 0.6000 | LR: 5.05e-05
  Ep  35/60 | T-Loss: 0.9387 | V-Loss: 0.4610 | V-Acc: 0.6400 | LR: 3.77e-05
  Ep  40/60 | T-Loss: 0.8679 | V-Loss: 0.6190 | V-Acc: 0.6000 | LR: 2.58e-05

  ⏹  Early stopping at epoch 44

✅ Training complete. Best val loss: 0.3921


## CELL 16 — Training Curve Plot

In [33]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(train_loss_hist, label='Train Loss', lw=2, color='#2196F3')
ax.plot(val_loss_hist,   label='Val Loss',   lw=2, color='#F44336')
ax.set(xlabel='Epoch', ylabel='Loss',
       title='Multi-Modal Contrastive Training — Loss Curve')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'training_curves.png', dpi=150)
plt.close(fig)
print(f'✅ Saved: {PLOT_DIR / "training_curves.png"}')

✅ Saved: C:\Users\Rathish K\Documents\ML\v1\plots\training_curves.png


## CELL 17 — Extract Embeddings for XGBoost (Batch, GPU)

In [71]:
def extract_embeddings(loader, model, device=DEVICE):
    """Extract concatenated [z_mri | z_tab] embeddings from DataLoader."""
    model.eval()
    all_emb, all_lbl = [], []
    with torch.no_grad(), autocast(enabled=USE_AMP):
        for mri_b, tab_b, lbl_b in tqdm(loader, desc='Extracting embeddings'):
            mri_b = mri_b.to(device, non_blocking=True)
            tab_b = tab_b.to(device, non_blocking=True)
            z_mri, z_tab = model.get_embeddings(mri_b, tab_b)
            emb = torch.cat([z_mri, z_tab], dim=1).cpu().float().numpy()
            all_emb.append(emb)
            all_lbl.extend(lbl_b.numpy())
    return np.concatenate(all_emb), np.array(all_lbl)

print('Extracting train embeddings...')
E_train, y_e_train = extract_embeddings(train_loader, fusion_model)
print('Extracting val embeddings...')
E_val,   y_e_val   = extract_embeddings(val_loader,   fusion_model)
print('Extracting test embeddings...')
E_test,  y_e_test  = extract_embeddings(test_loader,  fusion_model)

print(f'\n✅ Embeddings:')
print(f'  Train: {E_train.shape}  |  Val: {E_val.shape}  |  Test: {E_test.shape}')

Extracting train embeddings...


Extracting embeddings: 100%|██████████| 28/28 [00:00<00:00, 34.50it/s]


Extracting val embeddings...


Extracting embeddings: 100%|██████████| 7/7 [00:00<00:00, 74.90it/s]


Extracting test embeddings...


Extracting embeddings: 100%|██████████| 7/7 [00:00<00:00, 72.37it/s]


✅ Embeddings:
  Train: (112, 256)  |  Val: (25, 256)  |  Test: (25, 256)


## CELL 18 — XGBoost Classifier on Fused Embeddings

In [73]:
print('🌲 Training XGBoost on fused embeddings...')

# ── Defensive label remapping (shared across Cells 18 & 19) ──────────────
# XGBoost requires contiguous labels [0, 1, ..., k-1] with NO gaps.
# The MRI-matched subset (163 subjects) may have labels like [0,1,3]
# (gap at 2) if the full tabular 4-class encoding was used but some class
# is absent in the MRI subset.  We remap ALL label arrays here once.
_unique_labels = np.unique(y_e_train)                 # e.g. [0, 1, 3]
_expected      = np.arange(len(_unique_labels))       # e.g. [0, 1, 2]
if not np.array_equal(_unique_labels, _expected):
    # Map old integer → new contiguous integer
    _remap = {int(old): int(new) for new, old in enumerate(_unique_labels)}
    print(f'⚠️  Non-contiguous labels detected: {_unique_labels.tolist()}')
    print(f'   Remapping → {_expected.tolist()} (remap: {_remap})')
    # Remap embedding-split labels
    y_e_train = np.vectorize(_remap.get)(y_e_train).astype(np.int64)
    y_e_val   = np.vectorize(_remap.get)(y_e_val).astype(np.int64)
    y_e_test  = np.vectorize(_remap.get)(y_e_test).astype(np.int64)
    # Remap tabular-split labels (used by Cell 19)
    y_train_sm = np.vectorize(_remap.get)(y_train_sm).astype(np.int64)
    y_val      = np.vectorize(_remap.get)(y_val).astype(np.int64)
    y_test     = np.vectorize(_remap.get)(y_test).astype(np.int64)
    y_train    = np.vectorize(_remap.get)(y_train).astype(np.int64)
    # Rebuild CLASS_NAMES using the CANONICAL 4-class lookup so index 3 → 'AD'
    # This is safe regardless of how many classes Cell 5 ended up with.
    _CANONICAL_4 = ['CN', 'EMCI', 'LMCI', 'AD']
    CLASS_NAMES  = [_CANONICAL_4[i] for i in _unique_labels]
    N_CLASSES    = len(CLASS_NAMES)
    print(f'   Updated CLASS_NAMES: {CLASS_NAMES}  |  N_CLASSES: {N_CLASSES}')
else:
    _remap = None
    print(f'✅ Labels already contiguous: {_unique_labels.tolist()}')

xgb_model = xgb.XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1,
    early_stopping_rounds=30,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

xgb_model.fit(
    E_train, y_e_train,
    eval_set=[(E_val, y_e_val)],
    verbose=50,
)

xgb_pred  = xgb_model.predict(E_test)
xgb_proba = xgb_model.predict_proba(E_test)   # (N, N_CLASSES)

acc_xgb = accuracy_score(y_e_test, xgb_pred)
try:
    auc_xgb = roc_auc_score(y_e_test, xgb_proba,
                             multi_class='ovr', average='macro')
except Exception:
    auc_xgb = float('nan')
print(f'\n✅ XGBoost (emb) | Acc: {acc_xgb:.4f} | AUC: {auc_xgb:.4f}')


🌲 Training XGBoost on fused embeddings...
✅ Labels already contiguous: [0, 1, 2]
[0]	validation_0-mlogloss:1.04239
[50]	validation_0-mlogloss:0.61783
[95]	validation_0-mlogloss:0.62628

✅ XGBoost (emb) | Acc: 0.6400 | AUC: 0.8744


## CELL 19 — XGBoost on Raw Tabular Features (Baseline Comparison)

In [74]:
print('📊 Training XGBoost on raw tabular features (baseline)...')

# y_train_sm / y_val / y_test were already remapped to contiguous 0-based
# integers in Cell 18 (if needed). Safe to use directly here.
xgb_tab = xgb.XGBClassifier(
    n_estimators=500, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', random_state=SEED, n_jobs=-1,
    early_stopping_rounds=30,
    tree_method='hist',
    device='cuda' if torch.cuda.is_available() else 'cpu',
)
xgb_tab.fit(X_tab_train_sm, y_train_sm,
             eval_set=[(X_tab_val, y_val)], verbose=False)

xgb_tab_pred  = xgb_tab.predict(X_tab_test)
xgb_tab_proba = xgb_tab.predict_proba(X_tab_test)   # (N, N_CLASSES)

acc_xtab = accuracy_score(y_test, xgb_tab_pred)
try:
    auc_xtab = roc_auc_score(y_test, xgb_tab_proba,
                              multi_class='ovr', average='macro')
except Exception:
    auc_xtab = float('nan')
print(f'✅ XGBoost-Tab | Acc: {acc_xtab:.4f} | AUC: {auc_xtab:.4f}')


📊 Training XGBoost on raw tabular features (baseline)...
✅ XGBoost-Tab | Acc: 1.0000 | AUC: 1.0000


## CELL 20 — Shallow Neural Network Classifier (PyTorch, GPU)

In [75]:
class ShallowNN(nn.Module):
    """2-layer classifier on concatenated embeddings."""
    def __init__(self, in_dim, n_classes=N_CLASSES):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),    nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, n_classes)
        )
    def forward(self, x):
        return self.net(x)

snn_model = ShallowNN(E_train.shape[1]).to(DEVICE)
snn_opt   = torch.optim.Adam(snn_model.parameters(), lr=3e-4)
snn_ce    = nn.CrossEntropyLoss(weight=class_weights)

E_train_t = torch.from_numpy(E_train).float().to(DEVICE)
y_et      = torch.from_numpy(y_e_train).long().to(DEVICE)
E_val_t   = torch.from_numpy(E_val).float().to(DEVICE)
y_ev      = torch.from_numpy(y_e_val).long().to(DEVICE)

best_snn_loss = float('inf')
SNN_EPOCHS    = 80

for ep in range(SNN_EPOCHS):
    snn_model.train()
    snn_opt.zero_grad()
    logits = snn_model(E_train_t)
    loss   = snn_ce(logits, y_et)
    loss.backward()
    snn_opt.step()

    if ep % 20 == 0:
        snn_model.eval()
        with torch.no_grad():
            vl = snn_ce(snn_model(E_val_t), y_ev).item()
        if vl < best_snn_loss:
            best_snn_loss = vl
            torch.save(snn_model.state_dict(), MODEL_DIR / 'snn_best.pt')
        print(f'  SNN Ep {ep:3d} | Train: {loss.item():.4f} | Val: {vl:.4f}')

snn_model.load_state_dict(torch.load(MODEL_DIR / 'snn_best.pt', map_location=DEVICE))
snn_model.eval()
E_test_t = torch.from_numpy(E_test).float().to(DEVICE)
with torch.no_grad():
    snn_logits = snn_model(E_test_t)
    snn_proba  = F.softmax(snn_logits, dim=1).cpu().numpy()
    snn_pred   = snn_logits.argmax(1).cpu().numpy()

acc_snn = accuracy_score(y_e_test, snn_pred)
valid_cls = np.unique(y_e_test)
auc_snn = roc_auc_score(y_e_test, snn_proba[:, valid_cls] / snn_proba[:, valid_cls].sum(1, keepdims=True), multi_class='ovr', average='macro')
print(f'\n✅ SNN | Acc: {acc_snn:.4f} | AUC: {auc_snn:.4f}')

  SNN Ep   0 | Train: 1.0913 | Val: 1.0874
  SNN Ep  20 | Train: 1.0200 | Val: 1.0194
  SNN Ep  40 | Train: 0.9171 | Val: 0.9174
  SNN Ep  60 | Train: 0.7806 | Val: 0.8051

✅ SNN | Acc: 0.5200 | AUC: 0.6421


## CELL 21 — Fusion Model Test Evaluation

In [81]:
print('📊 Evaluating Fusion Model on test set...')

fusion_model.eval()
all_logits, all_labels = [], []

with torch.no_grad(), autocast(enabled=USE_AMP):
    for mri_b, tab_b, lbl_b in test_loader:
        mri_b = mri_b.to(DEVICE, non_blocking=True)
        tab_b = tab_b.to(DEVICE, non_blocking=True)
        logits = fusion_model(mri_b, tab_b)
        all_logits.append(F.softmax(logits, dim=1).cpu().float().numpy())
        all_labels.extend(lbl_b.numpy())

fusion_proba = np.concatenate(all_logits)
fusion_pred  = fusion_proba.argmax(1)
y_test_eval  = np.array(all_labels, dtype=np.int64)

# Remap y_test_eval to contiguous [0,1,...,k-1] using _remap from Cell 18.
# Use globals() which is reliable in Jupyter notebook scope.
_g = globals()
if '_remap' in _g and _g['_remap'] is not None:
    y_test_eval = np.vectorize(_g['_remap'].get)(y_test_eval).astype(np.int64)

acc_fus = accuracy_score(y_test_eval, fusion_pred)
try:
    auc_fus = roc_auc_score(
        y_test_eval,
        fusion_proba,
        multi_class='ovr',
        average='macro'
    )
except Exception:
    auc_fus = float('nan')
print(f'✅ Fusion Model | Acc: {acc_fus:.4f} | AUC: {auc_fus:.4f}')


📊 Evaluating Fusion Model on test set...
✅ Fusion Model | Acc: 0.0800 | AUC: 0.4902


## CELL 22 — Ensemble (Fusion + XGBoost + SNN) & Final Metrics

In [82]:
# Equal-weight ensemble — all three outputs are (N, N_CLASSES)
ens_proba = (fusion_proba + xgb_proba + snn_proba) / 3.0
ens_pred  = ens_proba.argmax(1)

def compute_metrics(y_true, y_pred, y_proba, name):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro')
    except Exception:
        auc = float('nan')
    print(f'  {name:20s} | Acc: {acc:.4f} | Prec: {prec:.4f} | '
          f'Rec: {rec:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')
    return dict(accuracy=acc, precision=prec, recall=rec, f1=f1, auc=auc)

print('='*75)
print('FINAL TEST METRICS')
print('='*75)
m_fus = compute_metrics(y_test_eval, fusion_pred,  fusion_proba,   'Fusion Model')
m_xgb = compute_metrics(y_e_test,    xgb_pred,     xgb_proba,      'XGBoost (emb)')
m_snn = compute_metrics(y_e_test,    snn_pred,      snn_proba,      'Shallow NN')
m_ens = compute_metrics(y_test_eval, ens_pred,      ens_proba,      '⭐ ENSEMBLE')
print('='*75)

print(f'\nDetailed Classification Report (Ensemble):')
print(classification_report(y_test_eval, ens_pred,
                             target_names=CLASS_NAMES,
                             labels=list(range(N_CLASSES)),
                             zero_division=0))


FINAL TEST METRICS
  Fusion Model         | Acc: 0.0800 | Prec: 0.0417 | Rec: 0.0833 | F1: 0.0556 | AUC: 0.4902
  XGBoost (emb)        | Acc: 0.6400 | Prec: 0.6794 | Rec: 0.6313 | F1: 0.6462 | AUC: 0.8744
  Shallow NN           | Acc: 0.5200 | Prec: 0.4087 | Rec: 0.4280 | F1: 0.3750 | AUC: 0.6421
  ⭐ ENSEMBLE           | Acc: 0.2800 | Prec: 0.4167 | Rec: 0.2500 | F1: 0.3095 | AUC: 0.8998

Detailed Classification Report (Ensemble):
              precision    recall  f1-score   support

          CN       1.00      0.50      0.67         6
        LMCI       0.67      0.50      0.57         8
          AD       0.00      0.00      0.00         0

   micro avg       0.28      0.50      0.36        14
   macro avg       0.56      0.33      0.41        14
weighted avg       0.81      0.50      0.61        14



## CELL 23 — Confusion Matrices & ROC Curves

In [85]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc as sklearn_auc

# ── Remap y_test_eval only (ground truth may still have old label 3) ──────
# ens_pred / fusion_pred are already 0-based (argmax of model probas).
# Only the DataLoader ground-truth labels need remapping.
_ul = np.unique(y_test_eval)
if not np.array_equal(_ul, np.arange(len(_ul))):
    _local_remap = {int(old): int(new) for new, old in enumerate(_ul)}
    y_test_eval  = np.vectorize(_local_remap.get)(y_test_eval).astype(np.int64)
    # DO NOT remap ens_pred — it is already in [0,1,...,N_CLASSES-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Confusion matrix ──────────────────────────────────────────────────────
cm = confusion_matrix(y_test_eval, ens_pred, labels=list(range(N_CLASSES)))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0],
            annot_kws={'size': 13})
axes[0].set(xlabel='Predicted', ylabel='True',
            title='Ensemble — Confusion Matrix')

# ── ROC curves ────────────────────────────────────────────────────────────
# y_test_eval is now contiguous [0,1,...,N_CLASSES-1].
# Use col_i for ALL indexing — always in-bounds.
classes_in_test = sorted(np.unique(y_test_eval))
y_bin  = label_binarize(y_test_eval, classes=classes_in_test)
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']

for col_i in range(len(classes_in_test)):
    cls_name = CLASS_NAMES[col_i]
    fpr, tpr, _ = roc_curve(y_bin[:, col_i], ens_proba[:, col_i])
    rval         = sklearn_auc(fpr, tpr)
    axes[1].plot(fpr, tpr, color=colors[col_i % 4], lw=2,
                 label=f'{cls_name} (AUC={rval:.3f})')

axes[1].plot([0,1],[0,1],'k--',lw=1)
axes[1].set(xlabel='False Positive Rate', ylabel='True Positive Rate',
            title='ROC Curves (One-vs-Rest)')
axes[1].legend(loc='lower right', fontsize=11)
axes[1].grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig(PLOT_DIR / 'evaluation_plots.png', dpi=150)
plt.close(fig)
print(f'✅ Saved: {PLOT_DIR / "evaluation_plots.png"}')


✅ Saved: C:\Users\Rathish K\Documents\ML\v1\plots\evaluation_plots.png


## CELL 24 — SHAP Explainability (Tabular Features)

In [86]:
print('🔍 Running SHAP analysis on XGBoost tabular model...')

import builtins
import shap

original_float = builtins.float
# We use a CLASS instead of a function so that Numpy's isinstance(val, (int, float)) doesn't crash!
class mock_float(float):
    def __new__(cls, val=0):
        if isinstance(val, str) and val.startswith('[') and val.endswith(']'):
            val = val.strip('[]').split(',')[0]
        return super().__new__(cls, val)

try:
    builtins.float = mock_float
    explainer = shap.TreeExplainer(xgb_tab)
finally:
    builtins.float = original_float  # Safely restore original float behavior

# By passing check_additivity=False, we tell SHAP not to panic about the tiny margin difference 
# caused by our math patch above.
shap_values = explainer.shap_values(X_tab_test, check_additivity=False)

# SHAP summary plot
fig, ax = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    shap_values if isinstance(shap_values, np.ndarray) else shap_values[len(shap_values)-1],
    X_tab_test,
    feature_names=FEATURE_NAMES,
    plot_type='bar',
    show=False,
    max_display=20,
)
plt.title('Top 20 Feature Importances (SHAP) — Tabular Model', fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'shap_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'✅ Saved: {PLOT_DIR / "shap_importance.png"}')

# SHAP beeswarm
shap.summary_plot(
    shap_values if isinstance(shap_values, np.ndarray) else shap_values[len(shap_values)-1],
    X_tab_test,
    feature_names=FEATURE_NAMES,
    show=False,
    max_display=20,
)
plt.title('SHAP Beeswarm — Feature Impact on AD Class', fontsize=13)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'✅ Saved: {PLOT_DIR / "shap_beeswarm.png"}')


🔍 Running SHAP analysis on XGBoost tabular model...
✅ Saved: C:\Users\Rathish K\Documents\ML\v1\plots\shap_importance.png
✅ Saved: C:\Users\Rathish K\Documents\ML\v1\plots\shap_beeswarm.png


## CELL 25 — Grad-CAM on MRI (PyTorch)

In [87]:
print('🔥 Generating Grad-CAM heatmaps...')

def compute_gradcam_3d(model, mri_tensor, target_class):
    """
    Manual Grad-CAM for 3D CNN.
    Hooks the last conv block of cnn_enc.
    Returns heatmap of shape (D, H, W).
    """
    model.eval()
    features, grads = {}, {}

    def fwd_hook(m, inp, out): features['f'] = out
    def bwd_hook(m, gi, go):   grads['g']    = go[0]

    # Hook last conv in block4
    h_fwd = model.cnn_enc.block4[0].block[0].register_forward_hook(fwd_hook)
    h_bwd = model.cnn_enc.block4[0].block[0].register_backward_hook(bwd_hook)

    mri_t = mri_tensor.to(DEVICE).unsqueeze(0).requires_grad_(True)
    tab_t = torch.zeros(1, n_tab_feats).to(DEVICE)  # neutral tabular

    logits = model(mri_t, tab_t)
    model.zero_grad()
    logits[0, target_class].backward()

    h_fwd.remove(); h_bwd.remove()

    feat = features['f'].detach().cpu().numpy()[0]   # (C, D, H, W)
    grad = grads['g'].detach().cpu().numpy()[0]      # (C, D, H, W)

    weights   = grad.mean(axis=(1, 2, 3))            # (C,)
    import scipy.ndimage
    cam       = (weights[:, None, None, None] * feat).sum(0)  # (D, H, W)
    cam       = np.maximum(cam, 0)                   # ReLU
    cam_norm  = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    
    # NEW: Upscale the low-res heatmap back to the original MRI dimensions!
    target_shape = mri_tensor.shape[1:]  # (D, H, W)
    zoom_factors = [t / c for t, c in zip(target_shape, cam_norm.shape)]
    cam_resized  = scipy.ndimage.zoom(cam_norm, zoom_factors, order=1)
    
    return cam_resized


# Generate for first test subject
try:
    sample_mri = test_ds[0][0]   # (1, D, H, W)
    pred_cls   = int(ens_pred[0])
    heatmap    = compute_gradcam_3d(fusion_model, sample_mri, pred_cls)

    # Visualise 3 axial slices
    mri_np = sample_mri.squeeze().numpy()
    mid    = np.array(MRI_TARGET_SHAPE) // 2
    slices = [mri_np[mid[0]], mri_np[:, mid[1]], mri_np[:, :, mid[2]]]
    hmaps  = [heatmap[mid[0]], heatmap[:, mid[1]], heatmap[:, :, mid[2]]]
    titles = ['Axial', 'Coronal', 'Sagittal']

    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    for i, (sl, hm, title) in enumerate(zip(slices, hmaps, titles)):
        axes[0, i].imshow(sl, cmap='gray'); axes[0, i].set_title(f'MRI — {title}'); axes[0, i].axis('off')
        axes[1, i].imshow(sl, cmap='gray')
        axes[1, i].imshow(hm, cmap='jet', alpha=0.5)
        axes[1, i].set_title(f'Grad-CAM — {title}'); axes[1, i].axis('off')

    fig.suptitle(f'Grad-CAM | Pred: {CLASS_NAMES[pred_cls]} | True: {CLASS_NAMES[y_test_eval[0]]}',
                 fontsize=13, fontweight='bold')
    fig.tight_layout()
    fig.savefig(PLOT_DIR / 'gradcam_mri.png', dpi=150)
    plt.close(fig)
    print(f'✅ Grad-CAM saved: {PLOT_DIR / "gradcam_mri.png"}')

except Exception as e:
    print(f'⚠️  Grad-CAM error: {e}')

🔥 Generating Grad-CAM heatmaps...
✅ Grad-CAM saved: C:\Users\Rathish K\Documents\ML\v1\plots\gradcam_mri.png


## CELL 26 — Missing Modality Robustness Test

In [88]:
print('🔬 Testing missing-modality robustness...')

fusion_model.eval()
scenarios = [
    ('Full (MRI + Tab)',  False, False),
    ('MRI only',          False, True),
    ('Tabular only',      True,  False),
]

print(f'  {"Scenario":<25} | Accuracy | AUC')
print('  ' + '-'*45)

for name, mri_miss, tab_miss in scenarios:
    preds, probas, trues = [], [], []
    with torch.no_grad(), autocast(enabled=USE_AMP):
        for mri_b, tab_b, lbl_b in test_loader:
            mri_b = mri_b.to(DEVICE); tab_b = tab_b.to(DEVICE)
            logits = fusion_model(mri_b, tab_b,
                                  mri_missing=mri_miss, tab_missing=tab_miss)
            proba  = F.softmax(logits, dim=1).cpu().float().numpy()
            probas.append(proba)
            preds.extend(proba.argmax(1))
            trues.extend(lbl_b.numpy())
    probas = np.concatenate(probas)
    trues  = np.array(trues)
    acc    = accuracy_score(trues, preds)
    try:
        a = roc_auc_score(trues, probas, multi_class='ovr', average='macro')
    except Exception:
        a = float('nan')
    print(f'  {name:<25} | {acc:.4f}   | {a:.4f}')

print('\n✅ Robustness test complete.')


🔬 Testing missing-modality robustness...
  Scenario                  | Accuracy | AUC
  ---------------------------------------------
  Full (MRI + Tab)          | 0.0800   | 0.4902
  MRI only                  | 0.0000   | 0.5649
  Tabular only              | 0.0000   | 0.4857

✅ Robustness test complete.


## CELL 27 — Per-Subject Prediction Report

In [90]:
report_rows = []
for i in range(len(y_test_eval)):
    row = {
        'Subject_Index'   : i,
        'True_Label'      : CLASS_NAMES[y_test_eval[i]],
        'Predicted_Label' : CLASS_NAMES[ens_pred[i]],
        'Correct'         : bool(y_test_eval[i] == ens_pred[i]),
        'AD_Risk_%'       : round(float(ens_proba[i, CLASS_NAMES.index('AD')]) * 100, 2),
    }
    for j, cls in enumerate(CLASS_NAMES):
        row[f'P_{cls}'] = round(float(ens_proba[i, j]), 4)
    report_rows.append(row)

report_df = pd.DataFrame(report_rows)
report_path = OUTPUT_DIR / 'per_subject_predictions.csv'
report_df.to_csv(report_path, index=False)
print(f'✅ Per-subject report saved: {report_path}')
print(report_df.head(10).to_string(index=False))

✅ Per-subject report saved: C:\Users\Rathish K\Documents\ML\v1\outputs\per_subject_predictions.csv
 Subject_Index True_Label Predicted_Label  Correct  AD_Risk_%   P_CN  P_LMCI   P_AD
             0         CN              AD    False      53.15 0.2615  0.2070 0.5315
             1         AD            LMCI    False      25.65 0.1833  0.5602 0.2565
             2         CN              CN     True      38.85 0.4353  0.1762 0.3885
             3         AD              AD     True      62.98 0.1816  0.1886 0.6298
             4         AD              AD     True      64.95 0.1851  0.1654 0.6495
             5       LMCI            LMCI     True      25.99 0.1778  0.5623 0.2599
             6       LMCI            LMCI     True      30.01 0.1736  0.5263 0.3001
             7         CN              CN     True      38.06 0.4481  0.1714 0.3806
             8       LMCI            LMCI     True      24.93 0.1927  0.5581 0.2493
             9         AD              AD     True      46.58

## CELL 28 — Save All Models & Artifacts

In [91]:
print('💾 Saving models and artifacts...')

# PyTorch models
torch.save(fusion_model.state_dict(), MODEL_DIR / 'fusion_model_final.pt')
torch.save(cnn_enc.state_dict(),      MODEL_DIR / 'cnn_encoder_final.pt')
torch.save(tab_enc.state_dict(),      MODEL_DIR / 'tab_encoder_final.pt')
torch.save(snn_model.state_dict(),    MODEL_DIR / 'shallow_nn_final.pt')

# XGBoost models
xgb_model.save_model(str(MODEL_DIR / 'xgboost_embedding.json'))
xgb_tab.save_model(str(MODEL_DIR   / 'xgboost_tabular.json'))

# Preprocessors
with open(MODEL_DIR / 'scaler.pkl',        'wb') as f: pickle.dump(scaler,        f)
with open(MODEL_DIR / 'imputer.pkl',       'wb') as f: pickle.dump(imputer,       f)
with open(MODEL_DIR / 'label_encoder.pkl', 'wb') as f: pickle.dump(le,            f)
with open(MODEL_DIR / 'feature_names.pkl', 'wb') as f: pickle.dump(FEATURE_NAMES, f)

# Metrics summary
metrics_all = {
    'Fusion Model' : m_fus,
    'XGBoost'      : m_xgb,
    'Shallow NN'   : m_snn,
    'Ensemble'     : m_ens,
}
with open(OUTPUT_DIR / 'metrics_summary.json', 'w') as f:
    json.dump(metrics_all, f, indent=2)

print(f'✅ All models saved to  : {MODEL_DIR}')
print(f'✅ Reports saved to     : {OUTPUT_DIR}')
print(f'✅ Plots saved to       : {PLOT_DIR}')

💾 Saving models and artifacts...
✅ All models saved to  : C:\Users\Rathish K\Documents\ML\v1\models
✅ Reports saved to     : C:\Users\Rathish K\Documents\ML\v1\outputs
✅ Plots saved to       : C:\Users\Rathish K\Documents\ML\v1\plots


## CELL 29 — Inference Function (Production Use)

In [92]:
def predict_patient(mri_nifti_path, tabular_raw_values: dict) -> dict:
    """
    Full inference for a new patient.
    Args:
        mri_nifti_path     : path to .nii/.nii.gz file, or None for tabular-only
        tabular_raw_values : dict of {feature_name: value} — raw, unscaled
    Returns:
        dict with Diagnosis, AD_Risk_%, Probabilities
    """
    import warnings; warnings.filterwarnings('ignore')

    # Build and scale tabular vector
    tab_vec = np.zeros(len(FEATURE_NAMES), dtype=np.float32)
    for k, v in tabular_raw_values.items():
        k_up = k.upper()
        if k_up in FEATURE_NAMES:
            tab_vec[FEATURE_NAMES.index(k_up)] = float(v)
    tab_scaled = scaler.transform(tab_vec[np.newaxis, :])[0].astype(np.float32)
    tab_t = torch.from_numpy(tab_scaled[np.newaxis, :]).float().to(DEVICE)

    mri_missing = (mri_nifti_path is None or str(mri_nifti_path).strip() == '')
    if not mri_missing:
        vol   = load_and_preprocess_mri(str(mri_nifti_path))
        mri_t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).float().to(DEVICE)
    else:
        mri_t = torch.zeros(1, 1, *MRI_TARGET_SHAPE, device=DEVICE)

    fusion_model.eval(); snn_model.eval()

    with torch.no_grad(), autocast(enabled=USE_AMP):
        fus_p = F.softmax(
            fusion_model(mri_t, tab_t, mri_missing=mri_missing),
            dim=1).cpu().float().numpy()[0]                        # (N_CLASSES,)
        xgb_p = xgb_tab.predict_proba(tab_scaled[np.newaxis, :])[0]  # (N_CLASSES,)
        z_m, z_t = fusion_model.get_embeddings(mri_t, tab_t)
        emb   = torch.cat([z_m, z_t], dim=1).float()
        snn_p = F.softmax(snn_model(emb.to(DEVICE)), dim=1).cpu().float().numpy()[0]

    ens_p  = (fus_p + xgb_p + snn_p) / 3.0
    pred_i = int(ens_p.argmax())

    ad_idx  = CLASS_NAMES.index('AD') if 'AD' in CLASS_NAMES else pred_i
    ad_risk = round(float(ens_p[ad_idx]) * 100, 2)

    result = {
        'Diagnosis'    : CLASS_NAMES[pred_i],
        'AD_Risk_%'    : ad_risk,
        'MRI_used'     : not mri_missing,
        'Probabilities': {cls: round(float(ens_p[i]), 4)
                          for i, cls in enumerate(CLASS_NAMES)},
    }

    print('='*55)
    print('   🧠 ALZHEIMER\'S RISK PREDICTION REPORT')
    print('='*55)
    print(f'  MRI used   : {"Yes" if not mri_missing else "No (tabular-only)"}')
    print(f'  Diagnosis  : {result["Diagnosis"]}')
    print(f'  AD Risk    : {result["AD_Risk_%"]} %')
    print(f'  Confidence : {max(result["Probabilities"].values())*100:.1f}%')
    print('  ' + '─'*40)
    for cls, prob in result['Probabilities'].items():
        bar = '█' * int(prob * 30)
        print(f'  {cls:4s}  {prob:.4f}  |{bar}')
    print('='*55)
    return result

print('✅ predict_patient() ready.')
print('   Usage: result = predict_patient(None, {"AGE":75, "MMSE":24, "CDRSB":2.5})')


✅ predict_patient() ready.
   Usage: result = predict_patient(None, {"AGE":75, "MMSE":24, "CDRSB":2.5})


## CELL 30 — Final Summary Dashboard

In [93]:
best_name = max(metrics_all, key=lambda k: metrics_all[k]['auc'])
bm = metrics_all[best_name]

print()
print('='*70)
print('       FINAL RESULTS — ALZHEIMER\'S MULTI-MODAL DETECTION')
print('='*70)
print(f'  Best Model    : {best_name}')
print(f'  Accuracy      : {bm["accuracy"]*100:.2f}%')
print(f'  Precision     : {bm["precision"]:.4f}')
print(f'  Recall        : {bm["recall"]:.4f}')
print(f'  F1-Score      : {bm["f1"]:.4f}')
print(f'  AUC (OvR)     : {bm["auc"]:.4f}')
print()
print(f'  Device        : {DEVICE}')
print(f'  MRI shape     : {MRI_TARGET_SHAPE}')
print(f'  Tab features  : {len(FEATURE_NAMES)}')
print(f'  Train subjects: {len(y_train)}')
print(f'  Test subjects : {len(y_test_eval)}')
print()
print('  Output files:')
for p in OUTPUT_DIR.iterdir():
    print(f'   • {p.name}')
for p in PLOT_DIR.iterdir():
    print(f'   • plots/{p.name}')
print('='*70)
print('  ✅ Project complete. All models and reports saved.')
print('='*70)


       FINAL RESULTS — ALZHEIMER'S MULTI-MODAL DETECTION
  Best Model    : Ensemble
  Accuracy      : 28.00%
  Precision     : 0.4167
  Recall        : 0.2500
  F1-Score      : 0.3095
  AUC (OvR)     : 0.8998

  Device        : cuda
  MRI shape     : (64, 64, 64)
  Tab features  : 263
  Train subjects: 113
  Test subjects : 25

  Output files:
   • metrics_summary.json
   • per_subject_predictions.csv
   • plots/evaluation_plots.png
   • plots/gradcam_mri.png
   • plots/shap_beeswarm.png
   • plots/shap_importance.png
   • plots/training_curves.png
  ✅ Project complete. All models and reports saved.


In [102]:
# =============================================================
# INTERACTIVE PATIENT INPUT
# Edit the values below and run this cell to get a prediction.
# All values are raw clinical numbers - no scaling needed.
# =============================================================

MY_PATIENT = {
    # Demographics
    'AGE'       : 75,       # Age in years
    'APOE4'     : 1,        # APOE4 alleles: 0, 1, or 2
    'PTEDUCAT'  : 14,       # Years of education

    # Cognitive scores
    'MMSE'      : 24,       # Mini-Mental State Exam (0-30, lower = worse)
    'CDRSB'     : 2.5,      # CDR Sum of Boxes (0=normal)
    'ADAS11'    : 18.0,     # ADAS-Cog 11 (higher = worse)
    'ADAS13'    : 24.0,     # ADAS-Cog 13
    'FAQ'       : 5,        # Functional Activities (0-30)

    # Memory tests
    'RAVLT_IMMEDIATE'  : 30,   # Word list recall (higher=better)
    'RAVLT_LEARNING'   : 3,
    'RAVLT_FORGETTING' : 5,
    'LDELTOTAL'        : 5,    # Logical memory delayed

    # Brain volumes (from MRI report if available, else leave as 0)
    'HIPPOCAMPUS': 6500,    # Hippocampal volume mm3
    'ENTORHINAL' : 3000,
    'VENTRICLES' : 38000,
    'WHOLEBRAIN' : 990000,

    # Vitals
    'VSBPSYS'   : 135,      # Systolic BP
    'VSBPDIA'   : 85,       # Diastolic BP
    'VSWEIGHT'  : 75,       # Weight kg
    'VSHEIGHT'  : 170,      # Height cm
}

MRI_PATH = None   # Set to r'C:\path\to\scan.nii.gz' or keep None

print('Running prediction for your patient...')
result = predict_patient(MRI_PATH, MY_PATIENT)


Running prediction for your patient...
   🧠 ALZHEIMER'S RISK PREDICTION REPORT
  MRI used   : No (tabular-only)
  Diagnosis  : AD
  AD Risk    : 45.61 %
  Confidence : 45.6%
  ────────────────────────────────────────
  CN    0.3479  |██████████
  LMCI  0.1960  |█████
  AD    0.4561  |█████████████


In [103]:
# Three contrasting patient profiles — run this to demonstrate to your mentor
DEMO_PATIENTS = {
    'Patient A (CN - Low Risk)': {
        'AGE':68,'APOE4':0,'MMSE':29,'CDRSB':0.0,'ADAS11':8.0,'ADAS13':11.0,
        'FAQ':0,'RAVLT_IMMEDIATE':48,'RAVLT_LEARNING':6,'LDELTOTAL':10,
        'HIPPOCAMPUS':7800,'ENTORHINAL':3900,'VENTRICLES':25000,
        'VSBPSYS':120,'VSBPDIA':78,'VSWEIGHT':72,
    },
    'Patient B (EMCI - Moderate Risk)': {
        'AGE':74,'APOE4':1,'MMSE':26,'CDRSB':1.5,'ADAS11':14.0,'ADAS13':18.0,
        'FAQ':3,'RAVLT_IMMEDIATE':32,'RAVLT_LEARNING':3,'LDELTOTAL':6,
        'HIPPOCAMPUS':6800,'ENTORHINAL':3200,'VENTRICLES':33000,
        'VSBPSYS':138,'VSBPDIA':88,'VSWEIGHT':78,
    },
    'Patient C (AD - High Risk)': {
        'AGE':82,'APOE4':2,'MMSE':19,'CDRSB':5.5,'ADAS11':28.0,'ADAS13':36.0,
        'FAQ':18,'RAVLT_IMMEDIATE':18,'RAVLT_LEARNING':1,'LDELTOTAL':2,
        'HIPPOCAMPUS':5200,'ENTORHINAL':2400,'VENTRICLES':55000,
        'VSBPSYS':155,'VSBPDIA':95,'VSWEIGHT':61,
    },
}

demo_results = {}
for pname, vals in DEMO_PATIENTS.items():
    print('\n' + '-'*55)
    print(f'🩺 {pname}')
    demo_results[pname] = predict_patient(None, vals)

# Summary table
print('\n' + '='*60)
print('  DEMO SUMMARY')
print('='*60)
print(f'  {"Patient":<35} | {"Diagnosis":6} | {"AD Risk %"}')
print('  ' + '-'*55)
for pname, res in demo_results.items():
    short = pname.split('(')[0].strip()
    print(f'  {short:<35} | {res["Diagnosis"]:6s}   | {res["AD_Risk_%"]:>6.1f}%')
print('='*60)



-------------------------------------------------------
🩺 Patient A (CN - Low Risk)
   🧠 ALZHEIMER'S RISK PREDICTION REPORT
  MRI used   : No (tabular-only)
  Diagnosis  : LMCI
  AD Risk    : 30.1 %
  Confidence : 51.2%
  ────────────────────────────────────────
  CN    0.1867  |█████
  LMCI  0.5123  |███████████████
  AD    0.3010  |█████████

-------------------------------------------------------
🩺 Patient B (EMCI - Moderate Risk)
   🧠 ALZHEIMER'S RISK PREDICTION REPORT
  MRI used   : No (tabular-only)
  Diagnosis  : AD
  AD Risk    : 47.8 %
  Confidence : 47.8%
  ────────────────────────────────────────
  CN    0.3114  |█████████
  LMCI  0.2106  |██████
  AD    0.4780  |██████████████

-------------------------------------------------------
🩺 Patient C (AD - High Risk)
   🧠 ALZHEIMER'S RISK PREDICTION REPORT
  MRI used   : No (tabular-only)
  Diagnosis  : CN
  AD Risk    : 32.66 %
  Confidence : 48.0%
  ────────────────────────────────────────
  CN    0.4805  |██████████████
  LMC